In [ ]:
import pandas as pd

<h2>Data Collection</h2>

In [ ]:
def load_dataset(file_path):
    return pd.read_csv(file_path, encoding='latin1')


In [ ]:
df = load_dataset('../dataset/TwitterDataset.csv')
df.head()

In [ ]:
# Remove unnecessary column
df.drop(columns=df.columns[0], inplace=True)
df

In [ ]:
# Renaming the 'Caption' to 'tweet'
df.rename(columns={'Caption': 'tweet'}, inplace=True)
df.rename(columns={'LABEL': 'sentiment'}, inplace=True)

label_mapping = {'negative': 0, 'neutral': 2, 'positive': 4}
df['sentiment'] = df['sentiment'].map(label_mapping)
df.head()

<h2>Pre Processing Data</h2>

* Cleansing Data: Removing @

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
# Removing @ from the tweet
def remove_usernames(text):
    return re.sub(r'@\w+', '', text)

def remove_hashtags(text):
    return re.sub(r'#', '', text)

def clean_text(text):
    # Removing URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Removing special characters and numbers
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', '', text)
    # Removing extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    return word_tokenize(text)

def case_folding(text):
    return text.lower()

def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [word for word in tokens if word not in stop_words]

def stem_tokens(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(word) for word in tokens]

def preprocess_text(text):
    text = remove_usernames(text)
    text = remove_hashtags(text)
    text = clean_text(text)
    text = case_folding(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)
    return ' '.join(tokens)

In [ ]:
df['cleaned_tweets'] = df['tweet'].apply(preprocess_text)

In [ ]:
df.head()

In [ ]:
# Only print data whose sentiment is neutral, I only need cleaned_tweets and sentiment columns
print(df[df['sentiment'] == 2][['cleaned_tweets', 'sentiment']])

<h2>Feature Extraction</h2>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from sklearn.pipeline import Pipeline
from sklearn import metrics



In [ ]:
X = df['cleaned_tweets']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# test_clf = Pipeline([
#     ('vect', CountVectorizer()),
#     ('tfidf', TfidfTransformer()),
#     ('clf', MultinomialNB()),
# ])

# test_clf.fit(X_train, y_train)

# # Predict on the test Data
# y_pred = test_clf.predict(X_test)

# #  Evaluate the model
# print(metrics.classification_report(y_test, y_pred))
# print(metrics.confusion_matrix(y_test, y_pred))
# print( "Accuracy" ,metrics.accuracy_score(y_test, y_pred))

In [ ]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred))

In [ ]:
# Make predictions on new data
# new_tweets = "I love this!", "This is awful.", "It's okay, not great.", "I hate this.", "Okay, Not Bad Not Good", "Just doing.", "this "]
# new_tweets_cleaned = [preprocess_text(tweet) for tweet in new_tweets]

# ? try
new_tweets = "I hate you but i love you"
new_tweets_cleaned = [preprocess_text(new_tweets)]

new_tweets_tfidf = vectorizer.transform(new_tweets_cleaned)
print(new_tweets_tfidf)
predictions = model.predict(new_tweets_tfidf)
predictions_proba = model.predict_proba(new_tweets_tfidf)

print(predictions_proba)
print(predictions)

<h2>Save the Model</h2>

In [ ]:
import joblib
# joblib.dump(model, 'model.pkl')

In [ ]:
joblib.dump(vectorizer, 'vectorizer.pkl')

In [ ]:
# Make predictions on new data
# new_tweets = "I love this!", "This is awful.", "It's okay, not great.", "I hate this.", "Okay, Not Bad Not Good", "Just doing.", "this "]
# new_tweets_cleaned = [preprocess_text(tweet) for tweet in new_tweets]

# ? try

model1 = joblib.load('model.pkl')
vectorizer1 = joblib.load('vectorizer.pkl')

new_tweets = "I normal This"


def predict(new_tweets):
    new_tweets_cleaned = [preprocess_text(new_tweets)]
    print("tweet cleaned",new_tweets_cleaned)

    new_tweets_tfidf = vectorizer1.transform(new_tweets_cleaned)
    print(new_tweets_tfidf)
    predictions = model1.predict(new_tweets_tfidf)
    print(predictions)
    
predict(new_tweets)
    

# FOR SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

In [ ]:
model = SVC(probability=True)
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred))

In [ ]:
# Make predictions on new data
# new_tweets = "I love this!", "This is awful.", "It's okay, not great.", "I hate this.", "Okay, Not Bad Not Good", "Just doing.", "this "]
# new_tweets_cleaned = [preprocess_text(tweet) for tweet in new_tweets]

# ? try
new_tweets = "normal"
new_tweets_cleaned = [preprocess_text(new_tweets)]

new_tweets_tfidf = vectorizer.transform(new_tweets_cleaned)
print(new_tweets_tfidf)
predictions = model.predict(new_tweets_tfidf)
predictions_proba = model.predict_proba(new_tweets_tfidf)
print(predictions_proba)
print(predictions)

# Saving the SVM Model

In [ ]:
import joblib
joblib.dump(model, 'svm.pkl')

### Hard Coded Naive Bayes

In [ ]:
# %run CustomNaiveBayes.py

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

class CustomNaiveBayes:
    def __init__(self):
        self.class_probs = {}
        self.feature_probs = {}
        self.classes = None
        self.vocab = None

    def fit(self, X, y):
        """
        Fit the Naive Bayes classifier according to the training data.

        Parameters:
        X: sparse matrix, shape (n_samples, n_features)
           Training data
        y: array-like, shape (n_samples,)
           Target labels
        """
        self.classes, class_counts = np.unique(y, return_counts=True)
        self.vocab = np.arange(X.shape[1])  # Feature indices
        
        # Initialize probabilities
        self.feature_probs = {c: np.zeros(len(self.vocab)) for c in self.classes}
        self.class_probs = {c: count / len(y) for c, count in zip(self.classes, class_counts)}

        # Count feature occurrences for each class
        for c in self.classes:
            class_data = X[y == c]
            word_counts = np.array(class_data.sum(axis=0)).flatten()
            self.feature_probs[c] = (word_counts + 1) / (word_counts.sum() + len(self.vocab))  # Add-one smoothing

    def predict(self, X):
        """
        Perform classification on an array of test vectors X.

        Parameters:
        X: sparse matrix, shape (n_samples, n_features)
           Test data

        Returns:
        array, shape (n_samples,)
            Predicted target values
        """
        predictions = []
        for doc in X:
            log_probs = {c: np.log(prob) for c, prob in self.class_probs.items()}
            word_counts = np.array(doc.sum(axis=0)).flatten()
            for c in self.classes:
                log_probs[c] += np.sum(np.log(self.feature_probs[c]) * word_counts)
            predictions.append(max(log_probs, key=log_probs.get))
        return np.array(predictions)


In [ ]:
model =  CustomNaiveBayes()
model.fit(X_train_tfidf, y_train)


In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred))

### Save the model

In [ ]:
import joblib
joblib.dump(model, 'custom_naive_bayes.pkl')

In [ ]:
model = joblib.load('custom_naive_bayes.pkl')


In [ ]:
# Make predictions on new data
# new_tweets = "I love this!", "This is awful.", "It's okay, not great.", "I hate this.", "Okay, Not Bad Not Good", "Just doing.", "this "]
# new_tweets_cleaned = [preprocess_text(tweet) for tweet in new_tweets]

# ? try
new_tweets = "I Love This, I hate this, normal this"
new_tweets_cleaned = [preprocess_text(new_tweets)]

new_tweets_tfidf = vectorizer.transform(new_tweets_cleaned)
# print(new_tweets_tfidf)
predictions = model.predict(new_tweets_tfidf)
print(predictions)

In [ ]:
joblib.dump(model, "custom.pkl")